In [1]:
import os
import sys
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from pathlib import Path

# Add src to path to import our modules
sys.path.append(os.path.abspath(os.path.join('..')))
from src.config import RAW_MAESTRO_DIR, PROCESSED_DIR
from src.preprocessing.midi_parser import process_single_file

print("Ready to extract piano-roll matrices.")

Ready to extract piano-roll matrices.


In [3]:
# We only want to process the files that fall into our 4 proxy genres
metadata_path = RAW_MAESTRO_DIR / "maestro-v3.0.0.csv"
df = pd.read_csv(metadata_path)

genre_mapping = {
    "Johann Sebastian Bach": "Baroque", "Domenico Scarlatti": "Baroque", "George Frideric Handel": "Baroque",
    "Wolfgang Amadeus Mozart": "Classical", "Joseph Haydn": "Classical", "Ludwig van Beethoven": "Classical",
    "Frédéric Chopin": "Romantic", "Franz Liszt": "Romantic", "Johannes Brahms": "Romantic",
    "Franz Schubert": "Romantic", "Robert Schumann": "Romantic", "Felix Mendelssohn": "Romantic",
    "Claude Debussy": "Impressionist_Modern", "Maurice Ravel": "Impressionist_Modern",
    "Alexander Scriabin": "Impressionist_Modern", "Sergei Rachmaninoff": "Impressionist_Modern",
    "Isaac Albéniz": "Impressionist_Modern"
}

df['proxy_genre'] = df['canonical_composer'].map(genre_mapping)
df_filtered = df.dropna(subset=['proxy_genre']).copy()
print(f"Total files to parse: {len(df_filtered)}")

Total files to parse: 1172


In [4]:
# Dictionaries to hold aggregated windows for each split
processed_data = {'train': [], 'validation': [], 'test': []}

# Process each file with a progress bar
for idx, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    midi_rel_path = row['midi_filename']
    split = row['split']
    
    full_path = RAW_MAESTRO_DIR / midi_rel_path
    
    if full_path.exists():
        # process_single_file converts to 16fs, binarizes, segments into 128-step windows, 
        # and removes windows with < 2% active cells
        windows = process_single_file(full_path)
        if windows.size > 0:
            processed_data[split].append(windows)
    else:
        print(f"Warning: File not found -> {full_path}")
        
# Concatenate lists into giant numpy arrays
for split in processed_data.keys():
    if len(processed_data[split]) > 0:
        processed_data[split] = np.concatenate(processed_data[split], axis=0)
        print(f"{split.capitalize()} set shape: {processed_data[split].shape}")
    else:
        processed_data[split] = np.array([])

  0%|          | 0/1172 [00:00<?, ?it/s]

Train set shape: (57441, 128, 88)
Validation set shape: (7166, 128, 88)
Test set shape: (7600, 128, 88)


In [5]:
# Save arrays to disk in data/processed/
for split, data_array in processed_data.items():
    if data_array.size > 0:
        save_path = PROCESSED_DIR / f"maestro_{split}_windows.npy"
        np.save(save_path, data_array)
        print(f"Saved {split} set to {save_path} ({data_array.nbytes / 1e6:.2f} MB)")
        
print("\nSuccess! Piano-roll extraction complete. The neural networks now have data to learn from.")

Saved train set to Y:\.college\CSE425\.project\music_generation_unsupervised\data\processed\maestro_train_windows.npy (2588.06 MB)
Saved validation set to Y:\.college\CSE425\.project\music_generation_unsupervised\data\processed\maestro_validation_windows.npy (322.87 MB)
Saved test set to Y:\.college\CSE425\.project\music_generation_unsupervised\data\processed\maestro_test_windows.npy (342.43 MB)

Success! Piano-roll extraction complete. The neural networks now have data to learn from.
